# makePredict_fromTIF — Google Colab

Versão Colab do script `makePredict_fromTIF_sortedByQuantity.py`.
Os TIFs de entrada e o modelo são lidos diretamente do Google Drive;
os TIFs classificados são salvos de volta no Drive.

## Metodologia

1. **Custom Keras Objects** — Registrados com `@register_keras_serializable` antes de carregar
   o modelo, garantindo que o Keras resolva os nomes serializados no arquivo `.keras`:
   - `dice_coef` / `dice_loss` (package `Custom`)
   - `focal_tversky_loss` / `boundary_loss` / `focal_tversky_boundary_loss` (package `RemoteSensing`)
   - `ResizeLike` — layer que redimensiona tensor A para dimensões espaciais de tensor B
   - `bce_dice_loss` / `hybrid_focal_loss` (package `RemoteSensing`)

2. **Inferência por patches sobrepostos** (`predict_tif`):
   - Lê o TIF completo com rasterio → transpõe (C,H,W)→(H,W,C) → normaliza ÷ 10 000
   - `patch_origins` gera posições (row, col) com stride < 256 garantindo cobertura das bordas
   - Roda `model.predict` em batches; acumula `pred_acc` e `weight_acc` para média
     ponderada nos pixels sobrepostos
   - Binariza com threshold e salva TIF com o mesmo CRS/transform original

3. **Recorte ao bounding box fotovoltaico** (`crop_to_fv_bbox`):
   - Projeta o array binário nos eixos: `rows_with_1 = np.any(data==1, axis=1)`,
     `cols_with_1 = np.any(data==1, axis=0)`
   - Determina `row_min/max` e `col_min/max` — fronteiras mais externas dos pixels positivos
   - Recorta e atualiza o geotransform com `rasterio.transform.from_origin`
   - Salva TIF recortado + linha no CSV acumulativo com as 2 coordenadas extremas
     (no CRS original e convertidas para WGS84 lon/lat)

---
**Todas as leituras e escritas usam o Google Drive montado em `/content/drive`.**

In [1]:
# ── Célula 1: Montar o Google Drive e instalar dependências ──────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install -q rasterio tqdm pyproj

Mounted at /content/drive


In [2]:
# ── Célula 2: PARÂMETROS DE ENTRADA ──────────────────────────────────────────
# Edite os valores abaixo antes de executar o pipeline.

# MODEL_PATH  = "/content/drive/MyDrive/DL_fotovoltaica/models_2026/best_5L_unet_efficientnetb7_20260520_2241.keras"
MODEL_PATH = "/content/drive/MyDrive/DL_fotovoltaica/models_2026/best_5L_deeplabv3plus_resnet101_20260722_2354_v3.keras"
INPUT_DIR   = "/content/drive/MyDrive/DS_FV_TIFs_scaled/PREDICT_V2"
OUTPUT_DIR  = "/content/drive/MyDrive/DL_fotovoltaica/tif_classificadas_2025"
sufixo = 'DL_RN101'

STRIDE      = 200    # stride em pixels entre patches (padrão: 200)
BATCH_SIZE  = 4      # patches por batch de inferência (padrão: 8)
THRESHOLD   = 0.5    # threshold de binarização; 0 = salvar probabilidade float32
OVERWRITE   = False  # True = reprocessa mesmo que o TIF de saída já exista

CROP_SUBDIR = "bbox_crops"  # subpasta dentro de OUTPUT_DIR para os TIFs recortados

print("Parâmetros carregados.")
print(f"  MODEL_PATH : {MODEL_PATH}")
print(f"  INPUT_DIR  : {INPUT_DIR}")
print(f"  OUTPUT_DIR : {OUTPUT_DIR}")
print(f"  STRIDE={STRIDE}px | BATCH_SIZE={BATCH_SIZE} | THRESHOLD={THRESHOLD}")

Parâmetros carregados.
  MODEL_PATH : /content/drive/MyDrive/DL_fotovoltaica/models_2026/best_5L_deeplabv3plus_resnet101_20260722_2354_v3.keras
  INPUT_DIR  : /content/drive/MyDrive/DS_FV_TIFs_scaled/PREDICT_V2
  OUTPUT_DIR : /content/drive/MyDrive/DL_fotovoltaica/tif_classificadas_2025
  STRIDE=200px | BATCH_SIZE=4 | THRESHOLD=0.5


In [3]:
# ── Célula 3: Imports e configuração de logging ───────────────────────────────
import re
import csv
import logging
import numpy as np
from pathlib import Path

import tensorflow as tf
for _gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(_gpu, True)
import keras
import keras.ops as kops
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print(f'TensorFlow: {tf.__version__}  |  Keras: {keras.__version__}')

import rasterio
import rasterio.transform
from pyproj import Transformer

try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    def tqdm(it, **kw):
        return it

# Logging: saída na célula + arquivo de log no Drive
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
LOG_FILE = Path(OUTPUT_DIR) / 'predict_colab.log'

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(str(LOG_FILE), encoding='utf-8'),
    ],
    force=True,
)
log = logging.getLogger('predict_colab')

gpus = tf.config.list_physical_devices('GPU')
log.info(f'GPU(s): {[g.name for g in gpus] or "nenhuma — CPU"}')

TensorFlow: 2.20.0  |  Keras: 3.13.2


2026-08-20 20:14:39,649 [INFO] GPU(s): ['/physical_device:GPU:0']


In [4]:
# ── Célula 4: Custom Keras Objects ───────────────────────────────────────────
# Devem ser registrados ANTES de tf.keras.models.load_model().
# Os packages e nomes precisam ser idênticos aos usados no treinamento.

# ── dice_coef / dice_loss  (package='Custom') ─────────────────────────────
@keras.utils.register_keras_serializable(package='Custom')
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = kops.reshape(kops.cast(y_true, 'float32'), [-1])
    y_pred_f = kops.reshape(kops.cast(y_pred, 'float32'), [-1])
    intersection = kops.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (kops.sum(y_true_f) + kops.sum(y_pred_f) + smooth)

@keras.utils.register_keras_serializable(package='Custom')
def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

# ── losses principais  (package='RemoteSensing') ──────────────────────────
@keras.utils.register_keras_serializable(package='RemoteSensing')
def focal_tversky_loss(y_true, y_pred, alpha=0.3, beta=0.7, gamma=1.25, smooth=1e-6):
    y_true   = kops.cast(y_true, 'float32')
    y_pred   = kops.cast(y_pred, 'float32')
    y_true_f = kops.reshape(y_true, [-1])
    y_pred_f = kops.reshape(y_pred, [-1])
    tp = kops.sum(y_true_f * y_pred_f)
    fp = kops.sum((1 - y_true_f) * y_pred_f)
    fn = kops.sum(y_true_f * (1 - y_pred_f))
    tversky_index = (tp + smooth) / (tp + alpha * fp + beta * fn + smooth)
    return kops.power((1 - tversky_index), gamma)

@keras.utils.register_keras_serializable(package='RemoteSensing')
def boundary_loss(y_true, y_pred, smooth=1e-6):
    y_true   = kops.cast(y_true, 'float32')
    y_pred   = kops.cast(y_pred, 'float32')
    dilated  =  tf.nn.max_pool2d( y_true, ksize=3, strides=1, padding='SAME')
    eroded   = -tf.nn.max_pool2d(-y_true, ksize=3, strides=1, padding='SAME')
    boundary = dilated - eroded
    p        = kops.clip(y_pred, 1e-7, 1.0 - 1e-7)
    bce      = -(y_true * kops.log(p) + (1 - y_true) * kops.log(1 - p))
    return kops.sum(bce * boundary) / (kops.sum(boundary) + smooth)

@keras.utils.register_keras_serializable(package='RemoteSensing')
def focal_tversky_boundary_loss(y_true, y_pred,
                                 alpha=0.3, beta=0.7, gamma=1.25,
                                 boundary_weight=0.85, smooth=1e-6):
    tversky  = focal_tversky_loss(y_true, y_pred, alpha=alpha, beta=beta,
                                   gamma=gamma, smooth=smooth)
    boundary = boundary_loss(y_true, y_pred, smooth=smooth)
    return tversky + boundary_weight * boundary

@keras.utils.register_keras_serializable(package='RemoteSensing')
def bce_dice_loss(y_true, y_pred):
    bce = keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)

@keras.utils.register_keras_serializable(package='RemoteSensing')
def hybrid_focal_loss(y_true, y_pred, alpha=0.25, gamma=2.0, smooth=1e-6):
    y_true = kops.cast(y_true, 'float32')
    y_pred = kops.cast(y_pred, 'float32')
    y_pred = kops.clip(y_pred, 1e-7, 1.0 - 1e-7)
    bce     = -(y_true * kops.log(y_pred) + (1 - y_true) * kops.log(1 - y_pred))
    p_t     = y_true * y_pred + (1 - y_true) * (1 - y_pred)
    alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
    focal_w = alpha_t * kops.power(1.0 - p_t, gamma)
    focal_bce = kops.mean(focal_w * bce)
    y_true_f = kops.reshape(y_true, [-1])
    y_pred_f = kops.reshape(y_pred, [-1])
    inter    = kops.sum(y_true_f * y_pred_f)
    d        = (2.0 * inter + smooth) / (kops.sum(y_true_f) + kops.sum(y_pred_f) + smooth)
    return focal_bce + kops.power(1.0 - d, gamma)

# ── ResizeLike layer ──────────────────────────────────────────────────────
@keras.utils.register_keras_serializable(package='RemoteSensing')
class ResizeLike(keras.layers.Layer):
    """Redimensiona x para o mesmo tamanho espacial de target via bilinear."""
    def call(self, inputs):
        x, target = inputs
        target_shape = tf.shape(target)
        return tf.image.resize(x, [target_shape[1], target_shape[2]])

    def compute_output_shape(self, input_shape):
        x_shape, target_shape = input_shape
        return (x_shape[0], target_shape[1], target_shape[2], x_shape[3])

CUSTOM_OBJECTS = {
    'dice_coef':                   dice_coef,
    'dice_loss':                   dice_loss,
    'focal_tversky_loss':          focal_tversky_loss,
    'boundary_loss':               boundary_loss,
    'focal_tversky_boundary_loss': focal_tversky_boundary_loss,
    'bce_dice_loss':               bce_dice_loss,
    'hybrid_focal_loss':           hybrid_focal_loss,
    'ResizeLike':                  ResizeLike,
}

print('Custom objects registrados:', list(CUSTOM_OBJECTS.keys()))

Custom objects registrados: ['dice_coef', 'dice_loss', 'focal_tversky_loss', 'boundary_loss', 'focal_tversky_boundary_loss', 'bce_dice_loss', 'hybrid_focal_loss', 'ResizeLike']


In [5]:
# ── Célula 5: Constantes ──────────────────────────────────────────────────────
NORM_FACTOR = 10_000.0   # int16 → float32 normalizado
PATCH_SIZE  = 256        # tamanho do patch quadrado (pixels)
N_BANDS     = 5          # número de bandas espectrais esperadas

print(f'NORM_FACTOR={NORM_FACTOR} | PATCH_SIZE={PATCH_SIZE}px | N_BANDS={N_BANDS}')

NORM_FACTOR=10000.0 | PATCH_SIZE=256px | N_BANDS=5


In [6]:
# ── Célula 6: Funções de modelo ───────────────────────────────────────────────

def _extract_model_key(model_path: Path) -> str:
    """Extrai chave identificadora do nome do arquivo .keras."""
    stem = model_path.stem
    key  = re.sub(r'_\d{8}_\d{4}$', '', stem)
    key  = re.sub(r'^best_(\d+[A-Za-z]+_)?', '', key)
    return key or stem


def load_model(model_path: str):
    """Carrega o modelo Keras com os custom objects registrados."""
    resolved = Path(model_path)
    if not resolved.exists():
        raise FileNotFoundError(f'Modelo não encontrado: {resolved}')
    log.info(f'Carregando modelo: {resolved}')
    model = tf.keras.models.load_model(str(resolved), custom_objects=CUSTOM_OBJECTS)
    log.info(f'Modelo carregado  |  input={model.input_shape}  output={model.output_shape}')
    return model


print('Funções de modelo definidas.')

Funções de modelo definidas.


In [7]:
# ── Célula 7: Patches e inferência ───────────────────────────────────────────

def patch_origins(size: int, patch: int, stride: int) -> list:
    """Origens dos patches em um eixo; o último patch sempre cobre a borda."""
    if size <= patch:
        return [0]
    origins = list(range(0, size - patch, stride))
    if not origins or origins[-1] + patch < size:
        origins.append(size - patch)
    return origins


def predict_tif(model, tif_path: Path, output_path: Path,
                stride: int, batch_size: int, threshold: float,
                overwrite: bool):
    """Executa inferência patch-a-patch e salva o TIF classificado."""
    if output_path.exists() and not overwrite:
        log.info(f'  já existe, pulando: {output_path.name}')
        return

    log.info(f'  lendo: {tif_path.name}')
    with rasterio.open(tif_path) as src:
        img       = src.read()       # (C, H, W) int16
        transform = src.transform
        crs       = src.crs

    img = np.transpose(img.astype(np.float32), (1, 2, 0)) / NORM_FACTOR
    H, W, C = img.shape

    if C != N_BANDS:
        log.warning(f'  bandas encontradas={C}, esperado={N_BANDS}. Usando as {N_BANDS} primeiras.')
        img = img[:, :, :N_BANDS]

    rows  = patch_origins(H, PATCH_SIZE, stride)
    cols  = patch_origins(W, PATCH_SIZE, stride)
    total = len(rows) * len(cols)
    log.info(f'  imagem {H}\u00d7{W}px | patches {len(rows)}\u00d7{len(cols)}={total} (stride={stride}px)')

    pred_acc   = np.zeros((H, W), dtype=np.float64)
    weight_acc = np.zeros((H, W), dtype=np.float64)

    batch_patches: list = []
    batch_coords:  list = []

    def flush():
        if not batch_patches:
            return
        arr   = np.stack(batch_patches)
        preds = model.predict(arr, verbose=0)
        for (r0, c0), pred in zip(batch_coords, preds[:, :, :, 0]):
            r1, c1 = min(r0 + PATCH_SIZE, H), min(c0 + PATCH_SIZE, W)
            ph, pw = r1 - r0, c1 - c0
            pred_acc  [r0:r1, c0:c1] += pred[:ph, :pw].astype(np.float64)
            weight_acc[r0:r1, c0:c1] += 1.0
        batch_patches.clear()
        batch_coords.clear()

    patch_iter = tqdm(
        [(r, c) for r in rows for c in cols],
        desc=tif_path.stem, unit='patch', disable=not HAS_TQDM,
    )
    for r0, c0 in patch_iter:
        r1, c1 = min(r0 + PATCH_SIZE, H), min(c0 + PATCH_SIZE, W)
        patch  = np.zeros((PATCH_SIZE, PATCH_SIZE, N_BANDS), dtype=np.float32)
        patch[:r1 - r0, :c1 - c0] = img[r0:r1, c0:c1]
        batch_patches.append(patch)
        batch_coords.append((r0, c0))
        if len(batch_patches) == batch_size:
            flush()

    flush()

    pred_map = (pred_acc / np.where(weight_acc == 0, 1.0, weight_acc)).astype(np.float32)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    if threshold > 0:
        data  = (pred_map > threshold).astype(np.uint8)
        dtype = np.uint8
        log.info(f'  threshold={threshold}  positivos={int(data.sum()):,}px')
    else:
        data  = pred_map
        dtype = np.float32

    with rasterio.open(
        output_path, 'w',
        driver='GTiff',
        height=H, width=W,
        count=1,
        dtype=dtype,
        crs=crs,
        transform=transform,
        compress='lzw',
    ) as dst:
        dst.write(data, 1)

    log.info(f'  salvo \u2192 {output_path.name}  ({H}\u00d7{W}px)')

print('patch_origins e predict_tif definidas.')

patch_origins e predict_tif definidas.


In [8]:
# ── Célula 8: crop_to_fv_bbox — Recorte ao bounding box fotovoltaico ─────────
#
# Lógica:
#   1. rows_with_1 = np.any(mask, axis=1)  → bool (H,)  — linhas com pixel positivo
#      cols_with_1 = np.any(mask, axis=0)  → bool (W,)  — colunas com pixel positivo
#   2. row_min/max e col_min/max são as fronteiras externas dos pixels positivos
#   3. Recorte: cropped = data[row_min:row_max+1, col_min:col_max+1]
#   4. Novo geotransform:
#        west  = transform.c + col_min * transform.a
#        north = transform.f + row_min * transform.e   (e < 0 → sul)
#   5. Salva TIF recortado e linha no CSV acumulativo

def crop_to_fv_bbox(output_path: Path, crop_dir: Path, csv_path: Path):
    """
    Recorta output_path ao bounding box dos pixels com valor 1.
    Salva o TIF recortado em crop_dir e adiciona uma linha em csv_path.
    """
    with rasterio.open(output_path) as src:
        data      = src.read(1)      # (H, W)
        transform = src.transform
        crs       = src.crs

    mask = (data == 1) if data.dtype == np.uint8 else (data > 0.5)

    rows_with_1 = np.any(mask, axis=1)   # (H,)
    cols_with_1 = np.any(mask, axis=0)   # (W,)

    if not rows_with_1.any():
        log.info(f'  sem pixels positivos em {output_path.name} — crop ignorado.')
        return

    row_min = int(np.argmax(rows_with_1))
    row_max = int(len(rows_with_1) - 1 - np.argmax(rows_with_1[::-1]))
    col_min = int(np.argmax(cols_with_1))
    col_max = int(len(cols_with_1) - 1 - np.argmax(cols_with_1[::-1]))

    cropped       = data[row_min : row_max + 1, col_min : col_max + 1]
    H_crop, W_crop = cropped.shape

    # Coordenadas dos cantos do bounding box no CRS original
    # transform.a > 0 (pixel em X, leste)  |  transform.e < 0 (pixel em Y, sul)
    # transform.c = X do canto superior-esquerdo original
    # transform.f = Y do canto superior-esquerdo original
    x_min = transform.c + col_min       * transform.a
    x_max = transform.c + (col_max + 1) * transform.a
    y_max = transform.f + row_min       * transform.e   # borda superior (maior Y)
    y_min = transform.f + (row_max + 1) * transform.e   # borda inferior (menor Y)

    new_transform = rasterio.transform.from_origin(
        west=x_min, north=y_max,
        xsize=transform.a, ysize=-transform.e,   # from_origin espera ysize positivo
    )

    crop_dir.mkdir(parents=True, exist_ok=True)
    cropped_path = crop_dir / f'crop_{output_path.name}'

    with rasterio.open(
        cropped_path, 'w',
        driver='GTiff',
        height=H_crop, width=W_crop,
        count=1,
        dtype=cropped.dtype,
        crs=crs,
        transform=new_transform,
        compress='lzw',
    ) as dst:
        dst.write(cropped, 1)

    log.info(f'  crop salvo \u2192 {cropped_path.name}  ({H_crop}\u00d7{W_crop}px)')

    # Conversão para WGS84 lon/lat
    try:
        proj = Transformer.from_crs(crs.to_epsg(), 4326, always_xy=True)
        lon_min, lat_min = proj.transform(x_min, y_min)
        lon_max, lat_max = proj.transform(x_max, y_max)
    except Exception as exc:
        log.warning(f'  Conversão WGS84 falhou ({exc}); usando NaN.')
        lon_min = lat_min = lon_max = lat_max = float('nan')

    write_header = not csv_path.exists()
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=[
            'arquivo', 'x_min', 'y_min', 'x_max', 'y_max',
            'lon_min', 'lat_min', 'lon_max', 'lat_max', 'crs',
        ])
        if write_header:
            writer.writeheader()
        writer.writerow({
            'arquivo': cropped_path.name,
            'x_min'  : round(x_min, 6),  'y_min': round(y_min, 6),
            'x_max'  : round(x_max, 6),  'y_max': round(y_max, 6),
            'lon_min': round(lon_min, 6), 'lat_min': round(lat_min, 6),
            'lon_max': round(lon_max, 6), 'lat_max': round(lat_max, 6),
            'crs'    : str(crs),
        })

    log.info(f'  bbox CRS  x=[{x_min:.4f}, {x_max:.4f}]  y=[{y_min:.4f}, {y_max:.4f}]')
    log.info(f'  bbox WGS84 lon=[{lon_min:.6f}, {lon_max:.6f}]  lat=[{lat_min:.6f}, {lat_max:.6f}]')

print('crop_to_fv_bbox definida.')

crop_to_fv_bbox definida.


In [10]:
# ── Célula 9: Pipeline principal ──────────────────────────────────────────────

model_path = Path(MODEL_PATH)
input_dir  = Path(INPUT_DIR)
output_dir = Path(OUTPUT_DIR)
crop_dir   = output_dir / CROP_SUBDIR
csv_path   = output_dir / 'fv_bboxes.csv'

model_key  = _extract_model_key(model_path)

log.info('=' * 60)
log.info(f'Modelo      : {model_path}')
log.info(f'Chave modelo: {model_key}')
log.info(f'Entrada     : {input_dir}')
log.info(f'Saída       : {output_dir}')
log.info(f'Crops bbox  : {crop_dir}')
log.info(f'CSV bbox    : {csv_path}')
log.info(f'Stride      : {STRIDE}px')
log.info(f'Batch size  : {BATCH_SIZE}')
log.info(f'Threshold   : {THRESHOLD}')
log.info('=' * 60)


def ja_processado(tif_path: Path, output_dir: Path) -> Path | None:
    """
    Retorna o Path do TIF de predição já salvo em output_dir para tif_path, ou None
    se ainda não foi processado. Usada para pular imagens já processadas sem sequer
    abrir o TIF de entrada nem rodar o modelo — só verifica se o arquivo de saída
    (mesmo padrão de nome usado no loop abaixo: '{stem}_pred.tif') já existe.
    """
    candidate = output_dir / f'{tif_path.stem}_pred.tif'
    return candidate if candidate.exists() else None


# ── Carrega modelo ────────────────────────────────────────────────────────
model = load_model(MODEL_PATH)
output_dir.mkdir(parents=True, exist_ok=True)

# ── Lista todos os TIFs da pasta de entrada ───────────────────────────────
tif_files = sorted(input_dir.glob('*.tif'))
if not tif_files:
    raise FileNotFoundError(f'Nenhum .tif encontrado em {input_dir}')

log.info(f'TIFs a processar: {len(tif_files)}')
for i, p in enumerate(tif_files, 1):
    log.info(f'  [{i:>3}/{len(tif_files)}] {p.name}')
log.info('=' * 60)

# ── Processa cada TIF ─────────────────────────────────────────────────────
# Nome de saída: mesmo nome do TIF original + sufixo _pred
# ex.: 00000000000000000005_2023.tif → 00000000000000000005_2023_pred.tif
processed = 0
skipped   = 0
for i, tif_path in enumerate(tif_files, 1):
    log.info(f'\n{"="*60}')
    log.info(f'[{i}/{len(tif_files)}] {tif_path.name}')

    output_path = output_dir / f'{tif_path.stem}_pred.tif'

    # Pula ANTES de chamar predict_tif/crop_to_fv_bbox — evita reabrir o TIF de
    # entrada e, principalmente, evita rodar crop_to_fv_bbox de novo sobre um TIF
    # já processado (o que duplicaria a linha correspondente em fv_bboxes.csv a
    # cada nova execução do notebook, já que predict_tif() sozinho só pulava a
    # inferência, sem impedir o recorte/CSV de rodar de novo em cima do resultado
    # antigo).
    if not OVERWRITE and ja_processado(tif_path, output_dir) is not None:
        log.info(f'  já processado, pulando: {output_path.name}')
        skipped += 1
        continue

    try:
        predict_tif(
            model      = model,
            tif_path   = tif_path,
            output_path= output_path,
            stride     = STRIDE,
            batch_size = BATCH_SIZE,
            threshold  = THRESHOLD,
            overwrite  = OVERWRITE,
        )

        # Recorte ao bounding box apenas para saídas binárias (threshold > 0)
        if THRESHOLD > 0 and output_path.exists():
            crop_to_fv_bbox(
                output_path = output_path,
                crop_dir    = crop_dir,
                csv_path    = csv_path,
            )

        processed += 1
    except Exception as exc:
        log.error(f'  Erro em {tif_path.name}: {exc}', exc_info=True)

log.info(f'\nConcluído. {processed} processado(s), {skipped} pulado(s) (já existiam), '
         f'de {len(tif_files)} TIF(s) total.')
log.info(f'Log: {LOG_FILE}')
if csv_path.exists():
    log.info(f'CSV bounding boxes: {csv_path}')

print('Pipeline finalizado.')


Streaming output truncated to the last 5000 lines.
2026-08-20 20:20:32,116 [INFO] 
2026-08-20 20:20:32,117 [INFO] [1455/2637] 0000000000000000073b_2025.tif
2026-08-20 20:20:32,118 [INFO]   já processado, pulando: 0000000000000000073b_2025_pred.tif
2026-08-20 20:20:32,119 [INFO] 
2026-08-20 20:20:32,120 [INFO] [1456/2637] 0000000000000000073c_2025.tif
2026-08-20 20:20:32,121 [INFO]   já processado, pulando: 0000000000000000073c_2025_pred.tif
2026-08-20 20:20:32,122 [INFO] 
2026-08-20 20:20:32,123 [INFO] [1457/2637] 00000000000000000741_2025.tif
2026-08-20 20:20:32,124 [INFO]   já processado, pulando: 00000000000000000741_2025_pred.tif
2026-08-20 20:20:32,124 [INFO] 
2026-08-20 20:20:32,125 [INFO] [1458/2637] 00000000000000000743_2025.tif
2026-08-20 20:20:32,126 [INFO]   já processado, pulando: 00000000000000000743_2025_pred.tif
2026-08-20 20:20:32,127 [INFO] 
2026-08-20 20:20:32,128 [INFO] [1459/2637] 00000000000000000745_2025.tif
2026-08-20 20:20:32,129 [INFO]   já processado, pulando:

Pipeline finalizado.


In [11]:
# ── Célula 10: Exportar lista de imagens já processadas (retomar no servidor local) ──
#
# Não depende de TensorFlow/Keras nem de GPU — só precisa da célula 1 (montar Drive)
# e da célula 2 (parâmetros) terem rodado. Serve para gerar, a qualquer momento
# (inclusive sem cota de GPU disponível no Colab), um CSV com o nome de todo TIF de
# entrada que já tem predição salva em OUTPUT_DIR — para um outro script (rodando no
# servidor local) ler esse CSV e continuar o processamento apenas das imagens que
# ainda faltam, sem precisar reprocessar o que já foi feito aqui no Colab.

import csv
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
input_dir  = Path(INPUT_DIR)

# Mesmo padrão de nome usado no pipeline principal (célula 9): '{stem}_pred.tif'
pred_files = sorted(output_dir.glob('*_pred.tif'))

processed_csv_path = output_dir / 'imagens_processadas.csv'
with open(processed_csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['tif_original', 'arquivo_pred'])
    for pred_path in pred_files:
        stem = pred_path.stem
        tif_original = stem[:-len('_pred')] if stem.endswith('_pred') else stem
        writer.writerow([f'{tif_original}.tif', pred_path.name])

print(f'{len(pred_files)} imagem(ns) já processada(s) em {output_dir}')
print(f'CSV salvo em: {processed_csv_path}')

# Contexto rápido: quantas ainda faltam, se INPUT_DIR estiver acessível neste momento.
if input_dir.exists():
    total_input = len(sorted(input_dir.glob('*.tif')))
    print(f'Total de TIFs em INPUT_DIR: {total_input}  |  faltando: {total_input - len(pred_files)}')

2623 imagem(ns) já processada(s) em /content/drive/MyDrive/DL_fotovoltaica/tif_classificadas_2025
CSV salvo em: /content/drive/MyDrive/DL_fotovoltaica/tif_classificadas_2025/imagens_processadas.csv
Total de TIFs em INPUT_DIR: 2637  |  faltando: 14
